# 🏆 MVP Copa do Mundo FIFA - Camada Bronze

## 1. Visão Geral do Notebook
Este notebook é responsável pela **Camada Bronze - import inicial dos dados brutos** da Copa do Mundo FIFA 2026. Ele realiza a leitura dos arquivos `.csv` armazenados no diretório do repositório Git, e cria as **Tabelas Gerenciadas (Delta Tables)** no **Unity Catalog**.

### 🎯 Objetivos:
1. Validar a integridade dos arquivos `.csv` na pasta de dados brutos (`../data/raw`).
2. Mapear e resolver os caminhos relativos no ambiente do Databricks Workspace.
3. Carregar os dados usando **PySpark**.
4. Salvar os dados brutos na tabela `copa_do_mundo.bronze.partidas_raw`.

---
> **Premissa Arquitetural (Medallion):** Os dados inseridos nesta camada são cópias fiéis do arquivo de origem (*as-is*). Nenhuma transformação, filtro ou limpeza de tipos é realizada nesta etapa. As correções serão aplicadas na camada **Silver**.

## 2. Configuração de Ambiente e Mapeamento de Diretórios
Nesta etapa, importamos os módulos necessários, identificamos a localização física do notebook dentro da estrutura do repositório Workspace e validamos o caminho relativo até a pasta de arquivos brutos (`../data/raw`).

In [0]:
import os

# 1. Mapeamento dos caminhos de execução
diretorio_notebook = os.getcwd()
caminho_relativo_raw = "../data/raw"
caminho_absoluto_raw = os.path.abspath(caminho_relativo_raw)

print(f"📍 Diretório do Notebook: {diretorio_notebook}")
print(f"📂 Diretório absoluto dos arquivos RAW: {caminho_absoluto_raw}")

## 3. Inventário dos Datasets de Origem
Mapeamos a relação dos 6 arquivos `.csv` esperados no repositório para garantir que a sincronização do Git Workspace ocorreu com sucesso antes de iniciar o processamento em lote.

| Arquivo Origem (.csv) | Tabela Alvo (Unity Catalog) | Conteúdo / Entidade |
| :--- | :--- | :--- |
| `match_prediction_features_bronze.csv` | `copa_do_mundo.bronze.match_prediction_features_bronze` | Métricas das partidas |
| `match_team_stats_bronze.csv` | `copa_do_mundo.bronze.match_team_stats_bronze` | Estatísticas por seleção e jogo |
| `matches_detailed_bronze.csv` | `copa_do_mundo.bronze.matches_detailed_bronze` | Detalhamento completo das partidas |
| `player_stats_bronze.csv` | `copa_do_mundo.bronze.player_stats_bronze` | Desempenho individual dos atletas |
| `referees_bronze.csv` | `copa_do_mundo.bronze.referees_bronze` | Dados sobre a equipe de arbitragem |
| `squads_and_players_bronze.csv` | `copa_do_mundo.bronze.squads_and_players_bronze` | Lista oficial de convocados por seleção |

## 4. Ingestão Automatizada em Lote (PySpark -> Unity Catalog)
Nesta etapa, definimos a governança de contexto (`USE CATALOG` e `USE SCHEMA`) e executamos um **loop de automação em Python**.

Para cada arquivo:
1. É realizada a leitura do CSV com cabeçalho (`header=True`) e detecção automática de tipos (`inferSchema=True`).
2. Os dados em memória são gravados no formato otimizado **Delta Lake** sob a estrutura `copa_do_mundo.bronze.<nome_da_tabela>`.
3. O modo de gravação selecionado é `overwrite`, garantindo a reexecução idempotente do pipeline.

In [0]:
# Lista com os nomes base dos arquivos sem a extensão
arquivos_copa = [
    "match_prediction_features_bronze",
    "match_team_stats_bronze",
    "matches_detailed_bronze",
    "player_stats_bronze",
    "referees_bronze",
    "squads_and_players_bronze",
]

# 1. Definição do contexto de governança no Unity Catalog
spark.sql("CREATE CATALOG IF NOT EXISTS copa_do_mundo")
spark.sql("USE CATALOG copa_do_mundo")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("USE SCHEMA bronze")

# 2. Pipeline automatizado de ingestão para as 6 tabelas
for nome_base in arquivos_copa:
    caminho_csv = f"file:{caminho_absoluto_raw}/{nome_base}.csv"
    nome_tabela_destino = f"copa_do_mundo.bronze.{nome_base}"

    print(f"🔄 Iniciando leitura do arquivo: {nome_base}.csv")

    # Leitura do CSV via PySpark
    df_temp = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ",")
        .load(caminho_csv)
    )

    # Persistência no Unity Catalog em formato Delta Lake
    df_temp.write.format("delta").mode("overwrite").saveAsTable(
        nome_tabela_destino
    )

    print(
        f"✅ Tabela gerenciada '{nome_tabela_destino}' gravada com sucesso!\n"
    )

print(
    "🎉 Ingestão concluída com sucesso! As 6 tabelas estão disponíveis no Unity Catalog."
)

## 5. Validação das Tabelas Criadas no Unity Catalog
Para encerrar a documentação do notebook, executamos uma consulta para listar as tabelas registradas no schema `bronze` e confirmar o volume final gerado.

In [0]:
# Exibe todas as tabelas registradas no schema atual
display(spark.sql("SHOW TABLES IN copa_do_mundo.bronze"))